In [ ]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler



In [ ]:
# Load the dataset (make sure the file is in the same directory or provide the correct path)
df = pd.read_excel("project_dataset.xlsx")
df.head()


In [ ]:
# Separate the input features (X), target variable (y), and country names for visualization
X = df.drop(columns=["Country", "BiodiversityIndex"])
y = df["BiodiversityIndex"]
countries = df["Country"]
feature_names = X.columns

In [ ]:
# Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test, countries_train, countries_test = train_test_split(
    X, y, countries, test_size=0.2, random_state=42
)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Create and train the Random Forest Regressor
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)


In [ ]:
# Predict biodiversity index for the test set
y_pred = rf_model.predict(X_test)

# Calculate and print MAE, RMSE, and R^2 score
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R² Score: {r2:.4f}")


In [ ]:
x = np.arange(len(countries_test))
height = 0.35  

plt.figure(figsize=(10, 16))  
plt.barh(x - height/2, y_test, height, label='Actual', color='seagreen', alpha=0.8)
plt.barh(x + height/2, y_pred, height, label='Predicted', color='skyblue', alpha=0.8)

for i, v in enumerate(y_test):
    plt.text(v + 0.01, i - height/2, f"{v:.4f}", va='center', fontsize=8)
for i, v in enumerate(y_pred):
    plt.text(v + 0.01, i + height/2, f"{v:.4f}", va='center', fontsize=8)

plt.yticks(x, countries_test, fontsize=8)
plt.xlabel("Biodiversity Index")
plt.title("Actual vs Predicted Biodiversity Index (Random Forest)")
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
error = np.abs(y_test - y_pred)
plt.figure(figsize=(18, 7))
plt.bar(countries_test, error, color="salmon")

for i, v in enumerate(error):
    plt.text(i, v + 0.005, f"{v:.4f}", color='black', ha='center', fontsize=8)

plt.xticks(rotation=90)
plt.ylabel("Absolute Error")
plt.xlabel("Country")
plt.title("Prediction Error (Absolute Error, Test Countries) - Random Forest")
plt.tight_layout()
plt.show()


In [ ]:
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
features_sorted = X.columns[indices]

plt.figure(figsize=(15, 12))
plt.barh(features_sorted, importances[indices], color="steelblue")
for i, v in enumerate(importances[indices]):
    plt.text(v + 0.005, i, f"{v:.5f}", color='black', va='center', fontsize=8)
plt.title("Feature Importance - Random Forest")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
# Group features into categories and calculate summed importance for each
feature_names = X.columns
importances = rf_model.feature_importances_

economy_features = ['GDP_PPP', 'HDI', 'Surfacearea']
natural_features = ['Latitude', 'Forestarea(%)', 'Waterresources(%)']  # Not: "lattitude" typo ile eşleşiyor
climate_features = [f for f in feature_names if f not in economy_features + natural_features]

category_scores = {
    'Economy': sum(importances[list(feature_names).index(f)] for f in economy_features),
    'Natural Environment': sum(importances[list(feature_names).index(f)] for f in natural_features),
    'Climate': sum(importances[list(feature_names).index(f)] for f in climate_features),
}

df_grouped = pd.DataFrame.from_dict(category_scores, orient='index', columns=['Importance'])

labels = df_grouped.index
sizes = df_grouped['Importance']
colors = ['#66c2a5', '#fc8d62', '#8da0cb']  

plt.figure(figsize=(6, 6))
plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140)
plt.title("Grouped Feature Importance by Category (Random Forest)")
plt.axis('equal')  
plt.tight_layout()
plt.show()
